# 01 — Data Preparation: Filter & Merge

**Pipeline stages:** 0A (Load) → 0B (Filter) → 0C (Merge)

This notebook prepares the unified analysis dataset:
1. Load raw prime contracts and subawards from USAspending
2. Filter primes: date range (2017-2024), exclude hardware PSC codes, exclude hardware keywords
3. Merge: decompose primes with subawards into constituent financial flows
4. Verify dollar integrity: merged total must equal filtered prime total

---

In [ ]:
import pandas as pd
import numpy as np
import os, sys, importlib.util

# Resolve project root (two levels up from this notebook)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(f'Project root: {PROJECT_ROOT}')

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

merge_mod = _import_module('create_merged_dataset',
    os.path.join(PROJECT_ROOT, 'notebooks', '01_prep', 'create_merged_dataset.py'))
filter_mod = _import_module('filter_primes',
    os.path.join(PROJECT_ROOT, 'notebooks', '01_prep', 'filter_primes.py'))


## 1. Load Raw Data

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', '01_raw')

primes_raw = merge_mod.load_primes(os.path.join(RAW_DIR, 'prime_awards.csv'))
subs_df    = merge_mod.load_subs(os.path.join(RAW_DIR, 'subawards.csv'))

In [ ]:
print('\nPrime columns:', list(primes_raw.columns))
print('Sub columns:  ', list(subs_df.columns))
print(f'\nPrimes date range: {primes_raw["Effective Date"].min()} to {primes_raw["Effective Date"].max()}')

## 2. Filter Primes

Three filters applied in sequence:
1. **Date**: Keep FY2017–2024 only (based on Effective Date year)
2. **PSC Code**: Remove 7 hardware PSC prefixes (7010, 7020, 7021, 7025, 7035, 5895, 5998)
3. **Keyword**: Remove contracts where description strongly matches hardware keywords with no service keywords

In [ ]:
primes_filtered, filter_summary = filter_mod.filter_primes(primes_raw, year_min=2017, year_max=2024)

In [ ]:
# Filtering funnel
stages = ['Raw', 'After date filter', 'After PSC filter', 'After keyword filter']
counts = [
    filter_summary['baseline_count'],
    filter_summary['after_date_count'],
    filter_summary['after_psc_count'],
    filter_summary['after_keyword_count'],
]
dollars = [
    filter_summary['baseline_dollars'] / 1e9,
    filter_summary['after_date_dollars'] / 1e9,
    filter_summary['after_psc_dollars'] / 1e9,
    filter_summary['after_keyword_dollars'] / 1e9,
]

funnel = pd.DataFrame({'Stage': stages, 'Contracts': counts, 'Spending ($B)': dollars})
funnel['% of Raw (contracts)'] = (funnel['Contracts'] / funnel['Contracts'].iloc[0] * 100).round(1)
funnel['% of Raw (dollars)']   = (funnel['Spending ($B)'] / funnel['Spending ($B)'].iloc[0] * 100).round(1)
print(funnel.to_string(index=False))

## 3. Merge Primes + Subawards

For each filtered prime:
- **Has subawards** → decompose into *retained* portion (prime keeps) + individual *subcontract* records
- **No subawards** → keep as *prime_no_subs*

Subawards are linked to primes by extracting the prime Award ID from the Subaward ID (first 6 underscore-separated segments). Subawards linked to primes that were excluded by filtering are automatically dropped.

In [ ]:
merged_df = merge_mod.create_merged_dataset(primes_filtered, subs_df)

### 3.1 Verification: Dollar Integrity

In [ ]:
passed = merge_mod.verify_merged_dataset(merged_df, primes_filtered, subs_df)
assert passed, 'Dollar integrity check FAILED'

### 3.2 Merged Dataset Summary

In [ ]:
merge_mod.print_merge_summary(merged_df)

In [ ]:
# Record type distribution
rt = merged_df.groupby('record_type').agg(
    records=('dollars', 'count'),
    dollars=('dollars', 'sum')
).sort_values('dollars', ascending=False)
rt['pct_records'] = (rt['records'] / rt['records'].sum() * 100).round(1)
rt['pct_dollars'] = (rt['dollars'] / rt['dollars'].sum() * 100).round(1)
rt['dollars_B'] = (rt['dollars'] / 1e9).round(2)
print(rt[['records', 'pct_records', 'dollars_B', 'pct_dollars']])

### 3.3 Subaward Linkage Analysis

Not all subawards link to filtered primes. Some link to primes that were excluded during filtering (pre-2017, hardware), and some may not link at all. Understanding this gap is important for interpreting the merged dataset.

In [ ]:
# Analyze subaward linkage
total_subs_raw = len(subs_df)
subs_in_merged = len(merged_df[merged_df['record_type'] == 'subcontract'])
subs_unlinked = total_subs_raw - subs_in_merged

# Break down unlinked subs: could they link to raw primes (pre-filter)?
raw_prime_ids = set(primes_raw['Award ID'].unique())
filtered_prime_ids = set(primes_filtered['Award ID'].unique())

subs_test = subs_df.copy()
subs_test['prime_award_id'] = subs_test['Subaward ID'].apply(
    lambda sid: merge_mod.extract_prime_award_id(sid, raw_prime_ids)
)

linked_to_any = subs_test['prime_award_id'].notna().sum()
linked_to_filtered = subs_test[subs_test['prime_award_id'].notna()].apply(
    lambda row: row['prime_award_id'] in filtered_prime_ids, axis=1
).sum()
linked_to_excluded = linked_to_any - linked_to_filtered

print(f'Total raw subawards:              {total_subs_raw:>8,}')
print(f'Linked to ANY prime:              {linked_to_any:>8,}  ({linked_to_any/total_subs_raw*100:.1f}%)')
print(f'  → Linked to filtered prime:     {linked_to_filtered:>8,}  ({linked_to_filtered/total_subs_raw*100:.1f}%)')
print(f'  → Linked to EXCLUDED prime:     {linked_to_excluded:>8,}  ({linked_to_excluded/total_subs_raw*100:.1f}%)')
print(f'  → Could not link to any prime:  {total_subs_raw - linked_to_any:>8,}  ({(total_subs_raw - linked_to_any)/total_subs_raw*100:.1f}%)')
print(f'\nSubawards in merged dataset:      {subs_in_merged:>8,}')
print(f'\nThe {linked_to_excluded:,} subs linked to excluded primes are appropriately dropped.')
print(f'They were associated with hardware or pre-2017 contracts excluded by filtering.')

In [ ]:
# Analyze the 205 primes where sub totals exceeded prime amount
# These required proportional scaling to maintain dollar integrity
prime_with_subs = merged_df[merged_df['record_type'].isin(['prime_retained', 'subcontract'])]
by_prime = prime_with_subs.groupby('original_prime_id').agg(
    merged_total=('dollars', 'sum'),
    n_records=('dollars', 'count')
)

# Compare merged total to original prime total
prime_originals = primes_filtered.set_index('Award ID')['Total Dollars Obligated']
by_prime['original_prime'] = by_prime.index.map(prime_originals)
by_prime = by_prime.dropna(subset=['original_prime'])

# Check for near-zero retained amounts (indicator of scaling)
retained = merged_df[merged_df['record_type'] == 'prime_retained']
zero_retained = (retained['dollars'] < 0.01).sum()

print(f'Primes decomposed into sub-flows: {len(by_prime):,}')
print(f'Primes with zero retained portion: {zero_retained:,}')
print(f'  (These are primes where subs consumed the entire prime amount)')
print(f'\nDollar integrity check:')
print(f'  Sum of merged records:  ${by_prime["merged_total"].sum()/1e9:.4f}B')
print(f'  Sum of original primes: ${by_prime["original_prime"].sum()/1e9:.4f}B')
print(f'  Difference:             ${abs(by_prime["merged_total"].sum() - by_prime["original_prime"].sum()):,.2f}')

## 4. Save Outputs

In [ ]:
OUT_FILTERED = os.path.join(PROJECT_ROOT, 'data', '02_processed', '01_filtered')
OUT_MERGED   = os.path.join(PROJECT_ROOT, 'data', '02_processed', '02_merged')
os.makedirs(OUT_FILTERED, exist_ok=True)
os.makedirs(OUT_MERGED, exist_ok=True)

filtered_path = os.path.join(OUT_FILTERED, 'prime_services_filtered.csv')
merged_path   = os.path.join(OUT_MERGED, 'merged_dataset.csv')

primes_filtered.to_csv(filtered_path, index=False)
print(f'Saved filtered primes: {filtered_path}  ({len(primes_filtered):,} rows)')

merged_df.to_csv(merged_path, index=False)
print(f'Saved merged dataset:  {merged_path}  ({len(merged_df):,} rows)')

---

**Next:** [02 — Baseline HHI Analysis](../02_base_analysis/02_baseline_hhi.ipynb)